In [ ]:
"""STEP 2: boot catalogue  ->  (the fake company)

Input  catalogue/boots.csv      the 11 catalogue boots (one row each, with specs)
       catalogue/parts.csv      each boot's parts: code, supplier, cost
       catalogue/suppliers.csv  the FIXED suppliers the catalogue boots use (planned test cases)
Output world/skus.csv, bom_lines.csv, suppliers.csv, customers.csv, orders.csv

SUPPLIERS: the fixed ones are kept, and extra suppliers are GENERATED for the parts that
can come from more than one place (toe caps, insoles, soles, linings...). Countries come
from the Component library's origin options; names come from Faker; every Canadian, US
or Mexican supplier gets a random declaration status (valid, expiring, expired, missing).

Each catalogue boot is copied into the world, plus a few VARIANTS made by changing it
the way a real product line changes: costs move, a part comes from another supplier,
a boot is made taller or shorter, a pac boot liner is bought finished. The same seed
always gives the same world, so everyone gets identical data.

Run:  python step2_world.py            (default 3 variants per boot)
      python step2_world.py 5          (5 variants per boot)
Needs: pip install faker
"""

from google.colab import drive
drive.mount('/content/drive', force_remount=True)


!pip install -q faker

import csv, pathlib, random, sys
from faker import Faker

from datetime import date, timedelta

# ---- FOLDERS: set the project location once, build everything from it ----
from pathlib import Path
REPO_ROOT        = Path('/content/drive/MyDrive/woodb-agent-project')   # your copy of the repo
CUSMA_ROOT       = REPO_ROOT / 'prototypes' / 'cusma_qualification'
CATALOGUE_FOLDER = CUSMA_ROOT / 'Catalogue'
ENGINE_FOLDER    = CUSMA_ROOT / 'Engine'
RULES_FOLDER     = CUSMA_ROOT / 'Rules'
WORLD_FOLDER     = CUSMA_ROOT / 'World'

!ls -la "{CATALOGUE_FOLDER}"

SEED = 894

# Parts that can come from different suppliers, and the countries they come from
# (Component library, "Origin options (synthetic)"). Used to generate suppliers.
SWAPPABLE = {
    "Safety toe": ["CN", "US", "MX", "DE"],
    "Insole": ["MX", "CN", "CA"],
    "Outsole": ["VN", "CN", "MX", "US"],          # soles bought finished
    "Lining": ["CN", "TW", "US"],
    "Puncture plate": ["DE", "CN", "US"],
    "Midsole": ["CA", "MX", "CN"],
    "Trims": ["CN", "CA"],
    "Packaging": ["CA", "US"],
}
NEW_PER_PART = 3                                    # generated suppliers per swappable part
CUSMA = {"CA", "US", "MX"}
# Declaration status for generated Canadian/US/Mexican suppliers: (status, share)
DECLARATION_MIX = [("valid", 0.60), ("expiring", 0.15), ("expired", 0.15), ("missing", 0.10)]
SUFFIX = {"CN": "Co., Ltd.", "VN": "Co., Ltd.", "TW": "Co., Ltd.", "DE": "GmbH", "IT": "S.r.l.",
          "MX": "S.A. de C.V.", "CA": "Inc.", "US": "LLC"}
WORD = {"Safety toe": "Safety Components", "Insole": "Comfort Insoles", "Outsole": "Sole Works",
        "Lining": "Textiles", "Puncture plate": "Protective Parts", "Midsole": "Footbed",
        "Trims": "Labels", "Packaging": "Packaging"}
CUSTOMERS = [
    ("CUS-201", "Lakehead Workwear Supply", "Duluth", "MN"),
    ("CUS-202", "Green Mountain Safety Outfitters", "Burlington", "VT"),
    ("CUS-203", "Cascade Marine & Outdoor", "Bellingham", "WA"),
    ("CUS-204", "Prairie Farm Supply Co.", "Fargo", "ND"),
    ("CUS-205", "Iron Range Industrial", "Hibbing", "MN"),
    ("CUS-206", "North Shore Fishing Gear", "Portland", "ME"),
]


def read(name):
    with open(CATALOGUE_FOLDER / name, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def write(name, rows):
    with open(WORLD_FOLDER / name, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)


def generate_suppliers(rng, fixed, parts):
    """Keep the fixed suppliers and generate NEW_PER_PART more for each swappable part."""
    fake = Faker("en_US") if Faker else None
    if fake:
        fake.seed_instance(894)
    surnames = ["Hale", "Moreau", "Ortega", "Lindqvist", "Tan", "Nguyen", "Becker", "Rossi", "Chen", "Dube", "Park", "Silva"]
    component_of = {}
    for p in parts:                                   # which part each fixed supplier provides
        component_of.setdefault(p["supplier_id"], p["component"])
    rows = [dict(s, component=component_of.get(s["supplier_id"], "")) for s in fixed]
    n = 101
    for comp, countries in SWAPPABLE.items():
        picks = rng.sample(countries, k=min(NEW_PER_PART, len(countries)))
        for country in picks:
            name = (fake.last_name() if fake else rng.choice(surnames)) + f" {WORD[comp]} {SUFFIX[country]}"
            if country in CUSMA:
                status = rng.choices([d for d, _ in DECLARATION_MIX], [w for _, w in DECLARATION_MIX])[0]
                expiry = {"valid": date(2027, 3, 1) + timedelta(days=rng.randint(0, 300)),
                          "expiring": date(2026, 11, 15) + timedelta(days=rng.randint(0, 75)),
                          "expired": date(2026, 6, 1) + timedelta(days=rng.randint(0, 150)),
                          "missing": None}[status]
                status = "valid" if status == "expiring" else status   # still valid today; runs out soon
            else:
                status, expiry = "not_applicable", None
            lang = "es" if country == "MX" else ("fr" if country == "CA" and rng.random() < 0.5 else "en")
            rows.append({"supplier_id": f"SUP-{n}", "name": name, "country": country, "supplies": comp,
                         "declaration_status": status, "declaration_expiry": expiry.isoformat() if expiry else "",
                         "language": lang, "component": comp})
            n += 1
    return rows


def make_variant(rng, boot, parts, suppliers, n):
    """Copy one catalogue boot and apply 1-2 realistic changes. Returns (boot, parts, description)."""
    b = dict(boot, sku=f"{boot['sku']}-V{n}", variant_of=boot["sku"])
    p = [dict(x, sku=b["sku"]) for x in parts]
    options = ["cost"]
    if any(x["component"] in SWAPPABLE for x in p):
        options.append("swap_supplier")
    if b["construction"] == "moulded" and b["toe_type"] != "steel":
        options.append("height")
    if any(x["component"] == "Liner" for x in p):
        options.append("bought_liner")
    changes = []
    for kind in rng.sample(options, k=min(len(options), rng.choice([1, 2]))):
        if kind == "cost":
            f = rng.uniform(0.85, 1.15)
            for x in p:
                x["cost_cad"] = f"{float(x['cost_cad']) * f:.2f}"
            b["labour_cad"] = f"{float(b['labour_cad']) * rng.uniform(0.9, 1.1):.2f}"
            changes.append(f"part costs x{f:.2f}")
        elif kind == "swap_supplier":
            line = rng.choice([x for x in p if x["component"] in SWAPPABLE])
            options_ = [s for s in suppliers.values() if s["component"] == line["component"] and s["supplier_id"] != line["supplier_id"]]
            new = rng.choice(options_)["supplier_id"]
            changes.append(f"{line['component']} from {new} ({suppliers[new]['country']}, declaration "
                           f"{suppliers[new]['declaration_status']}) instead of {line['supplier_id']}")
            line.update(supplier_id=new, origin_country=suppliers[new]["country"],
                        cost_cad=f"{float(line['cost_cad']) * rng.uniform(0.9, 1.1):.2f}")
        elif kind == "height":
            # stay realistic for the style: tall boots stay tall, short boots stay short
            h = rng.choice([15.5, 16.5, 18.0] if boot["covers_knee"] == "yes" else [6.0, 8.0, 12.0])
            b.update(height_in=f"{h}", covers_knee="yes" if h >= 15 else "no")
            changes.append(f"height {h} in.")
        elif kind == "bought_liner":
            line = next(x for x in p if x["component"] == "Liner")
            line.update(description="Removable felt liner, bought finished", hs_code="6406.10", cost_cad="5.20")
            changes.append("liner bought finished (6406.10)")
    b["style_name"] = f"{boot['style_name']} (variant {n})"
    b["tests"] = "Variant: " + "; ".join(changes)
    return b, p


def main(n_variants):
    rng = random.Random(SEED)
    boots, parts = read("boots.csv"), read("parts.csv")
    sup_rows = generate_suppliers(rng, read("suppliers.csv"), parts)
    suppliers = {s["supplier_id"]: s for s in sup_rows}
    WORLD_FOLDER.mkdir(exist_ok=True)

    world_boots, world_parts = [], []
    for boot in boots:
        bp = [x for x in parts if x["sku"] == boot["sku"]]
        world_boots.append(dict(boot, variant_of=boot.get("variant_of", "")))
        world_parts += bp
        for n in range(1, n_variants + 1):
            vb, vp = make_variant(rng, boot, bp, suppliers, n)
            world_boots.append(vb)
            world_parts += vp

    customers = [{"customer_id": c, "company": n, "city": ci, "state": st, "language": "en"} for c, n, ci, st in CUSTOMERS]
    orders, start = [], date(2026, 11, 2)
    for i, b in enumerate(world_boots):
        orders.append({"po_number": f"PO-{8001 + i}", "customer_id": rng.choice(CUSTOMERS)[0], "sku": b["sku"],
                       "quantity": rng.choice([100, 200, 300, 500]), "unit_price_usd": b["price_usd"],
                       "ship_date": (start + timedelta(days=rng.randint(0, 110))).isoformat()})

    write("skus.csv", world_boots)
    write("bom_lines.csv", world_parts)
    write("suppliers.csv", sup_rows)
    write("customers.csv", customers)
    write("orders.csv", orders)
    print(f"World written to world/: {len(world_boots)} boots ({len(boots)} catalogue + {len(world_boots) - len(boots)} variants), "
          f"{len(world_parts)} part lines, {len(sup_rows)} suppliers, {len(customers)} customers, {len(orders)} orders")
    gen = [x for x in sup_rows if int(x["supplier_id"][4:]) >= 101]
    print(f"Suppliers: {len(sup_rows) - len(gen)} fixed from the catalogue + {len(gen)} generated. Generated examples:")
    for x in gen[:6]:
        print(f"   {x['supplier_id']}  {x['name'][:40]:40} {x['country']}  {x['component']:14} {x['declaration_status']:15} {x['declaration_expiry']}")
    print("Boots:")
    for b in world_boots[:8]:
        print(f"   {b['sku']:11} {b['style_name'][:38]:38} {b['tests'][:70]}")
    print("   ...")


main(3)


In [ ]:
import pandas as pd
pd.set_option('display.max_rows', None, 'display.max_colwidth', None)
W = str(WORLD_FOLDER) + '/'

display(pd.read_csv(W + 'suppliers.csv'))   # all 46 suppliers
display(pd.read_csv(W + 'skus.csv')[['sku', 'style_name', 'height_in', 'tests']])   # all 44 boots and what changed

display(pd.read_csv(W + 'bom_lines.csv'))   # every part of every boot (376 rows)
display(pd.read_csv(W + 'orders.csv'))      # 44 orders
display(pd.read_csv(W + 'customers.csv'))   # 6 customers